# Snowflake 101 — Part 2

## Data that arrives as files, and pipelines that maintain themselves

In Part 1 the data was already a table. That is not how it usually arrives.

This notebook works with **terminal telemetry**: what a payment terminal reports back about itself. Model, firmware version, connectivity, battery, and a list of events such as `card_read_failed` or `printer_jam`. It arrives as JSON, because that is what devices emit.

By the end you will have:

1. Looked at a stage pointing outside your account, and one inside it
2. Queried JSON without defining a schema first
3. Turned a nested array into rows
4. Built a pipeline that keeps itself up to date
5. Asked Cortex Code to do a piece of the work for you

Allow about 40 minutes.

In [ ]:
%%sql -r ctx
-- Pins your role, database, schema and warehouse for the rest of the notebook.
USE ROLE ACCOUNTADMIN;
USE DATABASE FISERV_101_DB;
USE SCHEMA RAW;
USE WAREHOUSE FISERV_101_WH;

## 1. Stages

A **stage** is a location Snowflake can read files from. An *internal* stage is storage Snowflake manages for you. An *external* stage is a pointer at a bucket you or someone else owns, and Snowflake reads it in place without copying it first.

`PUBLIC_DATA_STAGE` points at `s3://sfquickstarts/`, a public bucket Snowflake maintains. You have no credentials for it and do not need any, because it is public. This is here to show you what an external stage *is*.

`LIST` returns one row per file, and that bucket holds a great many, so the pattern below narrows it to 300 PDFs. Snowflake is reading S3 directly to answer this.

In [ ]:
%%sql -r list_external
-- List an external stage. This reaches out to S3; no data is copied into Snowflake.
-- PATTERN keeps the output sane. Without it you get the whole bucket.
LIST @FISERV_101_DB.RAW.PUBLIC_DATA_STAGE PATTERN = '.*Invoices/.*[.]pdf';

In [ ]:
%%sql -r external_sample
-- RESULT_SCAN re-reads the previous cell's output as a table, so you can query it.
-- Useful whenever a SHOW or LIST returns more than you want to look at.
SELECT "name" AS FILE_PATH, "size" AS SIZE_BYTES
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY "size" DESC
LIMIT 10;

In [ ]:
%%sql -r list_internal
-- The internal stage holding the terminal telemetry you will actually work with.
LIST @FISERV_101_DB.RAW.TERMINAL_STAGE;

One file, roughly 900KB. That is what the setup produced, and it is enough: reading one JSON file off a stage teaches the same things as reading six.

Now read it. Note that you have not defined a single column.

In [ ]:
%%sql -r raw_json
-- Read the file straight off the stage. $1 is the whole JSON document per row.
-- No table, no column definitions, no load step.
SELECT $1 AS PAYLOAD
FROM @FISERV_101_DB.RAW.TERMINAL_STAGE (FILE_FORMAT => 'FISERV_101_DB.RAW.JSON_FF')
LIMIT 3;

## 2. VARIANT

`PAYLOAD` above is a **VARIANT**: a column that holds a whole JSON document, stored in a columnar format rather than as text. You reach into it with `:` for object keys and `[n]` for array positions, and you cast the result to the type you want with `::`.

Without the cast you get a VARIANT back, which prints with quotes around it and does not compare or sort the way you expect. That is the single most common mistake with semi-structured data in Snowflake.

In [ ]:
%%sql -r variant_paths
-- Reach into the document. Note the :: casts, and the nested device object.
SELECT
    $1:terminal_id::VARCHAR              AS TERMINAL_ID,
    $1:merchant_id::NUMBER               AS MERCHANT_ID,
    $1:reported_at::DATE                 AS REPORTED_AT,
    $1:device.model::VARCHAR             AS DEVICE_MODEL,
    $1:device.firmware_version::VARCHAR  AS FIRMWARE_VERSION,
    $1:device.connectivity::VARCHAR      AS CONNECTIVITY,
    $1:device.battery_percent::NUMBER    AS BATTERY_PERCENT,
    ARRAY_SIZE($1:events)                AS EVENT_COUNT
FROM @FISERV_101_DB.RAW.TERMINAL_STAGE (FILE_FORMAT => 'FISERV_101_DB.RAW.JSON_FF')
LIMIT 10;

Some `BATTERY_PERCENT` values are null. That is deliberate: a third of these terminals are mains powered and do not report a battery at all. Missing keys in JSON return null rather than failing, which is convenient and also how bad data gets through unnoticed.

## 3. FLATTEN

Every document has an `events` array. `ARRAY_SIZE` told you how many, but you cannot filter or group on the contents while they are still an array.

**FLATTEN** turns each array element into its own row. You use it as a lateral join, so each output row keeps its parent's columns alongside the element.

In [ ]:
%%sql -r flatten_events
-- One row per event rather than one row per terminal report.
-- e.VALUE is the array element; e.INDEX is its position.
SELECT
    t.$1:terminal_id::VARCHAR      AS TERMINAL_ID,
    e.INDEX                        AS EVENT_POSITION,
    e.VALUE:type::VARCHAR          AS EVENT_TYPE,
    e.VALUE:severity::VARCHAR      AS SEVERITY,
    e.VALUE:latency_ms::NUMBER     AS LATENCY_MS
FROM @FISERV_101_DB.RAW.TERMINAL_STAGE (FILE_FORMAT => 'FISERV_101_DB.RAW.JSON_FF') t,
     LATERAL FLATTEN(input => t.$1:events) e
LIMIT 12;

In [ ]:
%%sql -r event_summary
-- Now that events are rows, you can aggregate them. Which faults are worst?
SELECT
    e.VALUE:type::VARCHAR      AS EVENT_TYPE,
    e.VALUE:severity::VARCHAR  AS SEVERITY,
    COUNT(*)                   AS EVENT_COUNT,
    ROUND(AVG(e.VALUE:latency_ms::NUMBER), 1) AS AVG_LATENCY_MS
FROM @FISERV_101_DB.RAW.TERMINAL_STAGE (FILE_FORMAT => 'FISERV_101_DB.RAW.JSON_FF') t,
     LATERAL FLATTEN(input => t.$1:events) e
GROUP BY EVENT_TYPE, SEVERITY
ORDER BY EVENT_COUNT DESC;

In [ ]:
# Chart event volume by type, coloured by severity.
import matplotlib.pyplot as plt

df = event_summary.copy()
colour_for = {"info": "#29B5E8", "warning": "#FF9F36", "error": "#D45B90"}
colours = [colour_for.get(s, "#8A999E") for s in df["SEVERITY"]]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(df["EVENT_TYPE"], df["EVENT_COUNT"], color=colours)
ax.set_xlabel("Events")
ax.set_title("Terminal events by type (colour = severity)")
ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 4. From query to pipeline

That aggregate is useful, so it should not be something you re-run by hand.

First load the stage file into a table. Reading directly off a stage is fine for exploring, but a pipeline needs a table it can track changes on.

In [ ]:
%%sql -r load_table
-- Land the raw documents in a table, one VARIANT column, nothing reshaped yet.
-- Keeping the raw payload means you can change your mind about the shape later.
CREATE OR REPLACE TABLE FISERV_101_DB.RAW.TERMINAL_TELEMETRY AS
SELECT $1 AS PAYLOAD
FROM @FISERV_101_DB.RAW.TERMINAL_STAGE (FILE_FORMAT => 'FISERV_101_DB.RAW.JSON_FF');

In [ ]:
%%sql -r loaded_count
-- Confirm what landed.
SELECT COUNT(*) AS DOCUMENTS_LOADED FROM FISERV_101_DB.RAW.TERMINAL_TELEMETRY;

### Dynamic tables

A **dynamic table** is defined by a query, not by a load process. You declare the result you want and a `TARGET_LAG` — how stale you are willing to let it get — and Snowflake works out what to re-compute and when. There is no task to schedule, no merge statement to write, and no ordering to get right.

Where it can, Snowflake refreshes **incrementally**: it processes only the rows that changed rather than rebuilding the whole thing. Where the query is too complex for that, it falls back to a full refresh. You will check which one you got, because the difference matters at volume and Snowflake does not always choose what you assumed.

In [ ]:
%%sql -r create_dt
-- A dynamic table over the raw telemetry. TARGET_LAG is the promise Snowflake
-- makes you: this will never be more than a minute behind the source.
CREATE OR REPLACE DYNAMIC TABLE FISERV_101_DB.ANALYTICS.TERMINAL_EVENT_SUMMARY
  TARGET_LAG = '1 minute'
  WAREHOUSE = FISERV_101_WH
AS
SELECT
    t.PAYLOAD:device.model::VARCHAR      AS DEVICE_MODEL,
    e.VALUE:type::VARCHAR                AS EVENT_TYPE,
    e.VALUE:severity::VARCHAR            AS SEVERITY,
    COUNT(*)                             AS EVENT_COUNT,
    ROUND(AVG(e.VALUE:latency_ms::NUMBER), 1) AS AVG_LATENCY_MS
FROM FISERV_101_DB.RAW.TERMINAL_TELEMETRY t,
     LATERAL FLATTEN(input => t.PAYLOAD:events) e
GROUP BY DEVICE_MODEL, EVENT_TYPE, SEVERITY;

In [ ]:
%%sql -r dt_contents
-- The dynamic table already has data. Creating it ran the first refresh.
SELECT DEVICE_MODEL, EVENT_TYPE, SEVERITY, EVENT_COUNT, AVG_LATENCY_MS
FROM FISERV_101_DB.ANALYTICS.TERMINAL_EVENT_SUMMARY
ORDER BY EVENT_COUNT DESC
LIMIT 10;

In [ ]:
%%sql -r show_dt
-- SHOW exposes the refresh mode Snowflake chose, which the DDL does not tell you.
SHOW DYNAMIC TABLES LIKE 'TERMINAL_EVENT_SUMMARY' IN SCHEMA FISERV_101_DB.ANALYTICS;

In [ ]:
%%sql -r dt_refresh_mode
-- Which refresh mode did you actually get, and why?
-- REFRESH_MODE should say INCREMENTAL. REFRESH_MODE_REASON is null when
-- Snowflake gave you what you asked for; it only fills in to explain a
-- downgrade to FULL. A null here is good news.
SELECT "name" AS DYNAMIC_TABLE,
       "target_lag" AS TARGET_LAG,
       "refresh_mode" AS REFRESH_MODE,
       "refresh_mode_reason" AS REFRESH_MODE_REASON
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

### Prove it maintains itself

First establish a baseline. In the seed data, Clover Flex terminals only ever report `warning` severity, never `error`, so this next cell returns **no rows at all**.

That absence is the thing you are going to change.

In [ ]:
%%sql -r baseline
-- Baseline: error-severity events on Clover Flex. Expect zero rows.
SELECT DEVICE_MODEL, EVENT_TYPE, SEVERITY, EVENT_COUNT, AVG_LATENCY_MS
FROM FISERV_101_DB.ANALYTICS.TERMINAL_EVENT_SUMMARY
WHERE DEVICE_MODEL = 'Clover Flex' AND SEVERITY = 'error'
ORDER BY EVENT_TYPE;

Now add one new terminal report to the **source table**. Do not touch the dynamic table.

In [ ]:
%%sql -r insert_new
-- One new terminal report, with two events, both errors on a new firmware.
INSERT INTO FISERV_101_DB.RAW.TERMINAL_TELEMETRY (PAYLOAD)
SELECT OBJECT_CONSTRUCT(
    'terminal_id', 'TRM999999',
    'merchant_id', 4242,
    'reported_at', '2025-09-30',
    'device', OBJECT_CONSTRUCT('model', 'Clover Flex',
                               'firmware_version', '5.0.1',
                               'connectivity', 'cellular',
                               'battery_percent', 12),
    'events', ARRAY_CONSTRUCT(
        OBJECT_CONSTRUCT('type', 'card_read_failed', 'severity', 'error', 'latency_ms', 1800),
        OBJECT_CONSTRUCT('type', 'network_drop',     'severity', 'error', 'latency_ms', 2400))
);

In [ ]:
%%sql -r force_refresh
-- TARGET_LAG is one minute, so this would happen on its own shortly.
-- Refreshing manually means you do not have to wait for it during the session.
ALTER DYNAMIC TABLE FISERV_101_DB.ANALYTICS.TERMINAL_EVENT_SUMMARY REFRESH;

In [ ]:
%%sql -r dt_after
-- The same query as the baseline. Two rows now exist where there were none.
-- You wrote no update logic, no MERGE, and no task.
SELECT DEVICE_MODEL, EVENT_TYPE, SEVERITY, EVENT_COUNT, AVG_LATENCY_MS
FROM FISERV_101_DB.ANALYTICS.TERMINAL_EVENT_SUMMARY
WHERE DEVICE_MODEL = 'Clover Flex' AND SEVERITY = 'error'
ORDER BY EVENT_TYPE;

In [ ]:
%%sql -r refresh_history
-- The refresh log. REFRESH_ACTION tells you what each refresh actually did.
-- Expect CREATION, then possibly one or more SCHEDULED rows showing NO_DATA
-- (target lag came round and there was nothing to do), then your MANUAL one
-- showing INCREMENTAL. NO_DATA refreshes are near-free, which is why a tight
-- TARGET_LAG on a quiet table is not the cost problem people assume.
SELECT NAME, STATE, REFRESH_ACTION, REFRESH_TRIGGER,
       DATA_TIMESTAMP, REFRESH_START_TIME
FROM TABLE(FISERV_101_DB.INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(
    NAME => 'FISERV_101_DB.ANALYTICS.TERMINAL_EVENT_SUMMARY'))
ORDER BY REFRESH_START_TIME;

## Hand-off — your first job for Cortex Code

**Why:** you have written every line of SQL in this notebook so far. Cortex Code (CoCo) is built into Snowsight and can write and run it for you against your actual account, with your actual object names. From here on you will use it for the parts that are typing rather than thinking.

This is deliberately a task you could do yourself, so you can judge the answer.

**Steps**

1. In Snowsight, open a **Workspace** from the left-hand nav, then open the **Cortex Code** panel.
2. Type this prompt exactly:

   > In FISERV_101_DB.RAW.TERMINAL_TELEMETRY the PAYLOAD column is a VARIANT holding terminal telemetry. Find the firmware versions with the highest proportion of error-severity events. Show firmware version, total events, error events, and error rate as a percentage, worst first. Exclude firmware versions with fewer than 50 total events.

3. Let it run the query it writes.
4. Read the SQL it produced. Check three things: did it use `LATERAL FLATTEN`, did it cast with `::`, and is the error rate denominator *total events* rather than total terminals?
5. Now ask a follow-up in the same conversation:

   > Is 5.0.1 an outlier, and how confident should I be given how few reports it has?

**What you should see:** a working query, and a note that firmware `5.0.1` has a very high error rate on a single report, which is not evidence of anything. The follow-up matters more than the first answer: the number is easy and the caveat is the part that stops someone acting on one data point.

**Docs:** https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-code

**Return here** when you have both answers.

## What you did

- Listed an **external stage** over a public bucket and an **internal stage** in your own account
- Queried JSON straight off a stage with no schema and no load step
- Used **VARIANT** path notation and casts, and saw what a missing key returns
- Used **FLATTEN** to turn a nested array into rows you can group on
- Built a **dynamic table** with a one-minute `TARGET_LAG`, then proved it picked up a new record without you writing any update logic
- Checked the **refresh mode** rather than assuming it
- Gave Cortex Code a task and checked its work

## Where this goes

Tomorrow's session 4 uses the same dynamic table pattern over 30 million fee lines in three layers, and the refresh-mode question stops being academic.

**Part 3** covers who is allowed to see what, and what you can do with the free-text fields you have been ignoring so far.